# 5 Your own search, and a negative result you produced yourself

Notebook 04 ran one search and read it for you. Here you drive.

**The deliverable is a short report** — half a page is plenty — containing at least one
**negative result that you produced and are willing to stand behind**. Not a failure you
apologise for: a hypothesis you stated, tested cheaply, and found did not pay. That habit
is the entire reason this lesson exists, and it is the thing being graded.

A report that says "I tried six things and here is the best one" is worth less than one
that says "I expected X, measured it, got Y, and here is the seed spread that makes me
believe it".

Everything runs on **DTD** — 47 texture classes, roughly twenty points of headroom, and an
encoder axis that actually moves.

In [ ]:
import os
import warnings

os.environ.setdefault("RAY_ENABLE_UV_RUN_RUNTIME_ENV", "0")     # see notebook 04
warnings.filterwarnings("ignore", message=".*pkg_resources.*")

import sys
import time
from pathlib import Path

import datasets
import matplotlib.pyplot as plt
import numpy as np
import torch
from loguru import logger
from torch import nn
from torch.utils.data import DataLoader
from vectormesh import VectorCache
from vectormesh.components import GaussianNoise, NeuralNet, Serial
from vectormesh.data import Collate, OneHot

from notebooktester import param

logger.remove()
logger.add(sys.stderr, level="WARNING")
datasets.disable_progress_bars()

EPOCHS = param(15, test=1)
N_TRIALS = param(16, test=2)
SEEDS = param(3, test=2)

ENCODERS = ["dinov2-small", "resnet-18", "mobilenet_v2_1.0_224"]

In [ ]:
def load(encoder: str, split: str) -> VectorCache:
    return VectorCache.from_hub(f"pttrn-io/dtd-{encoder}", split=split)


def loaders(encoder: str, batch_size: int = 64, n_train: int | None = None):
    # n_train shrinks the training set for the data-size questions below.
    # The shuffle is NOT optional -- see the cell after this one.
    out = []
    for split in ("train", "test"):
        cache = load(encoder, split)
        ds = cache.dataset
        if n_train is not None and split == "train":
            ds = ds.shuffle(seed=0).select(range(n_train))
        n_classes = cache.features["label"].num_classes
        oh = OneHot(num_classes=n_classes, label_col="label", target_col="onehot")
        col = Collate(embedding_col="embed", target_col="onehot", padder=torch.stack)
        out.append(DataLoader(ds.map(oh), batch_size=batch_size,
                              shuffle=(split == "train"), collate_fn=col))
    return out


def run(model, train_dl, test_dl, lr: float = 1e-3, epochs: int = EPOCHS) -> float:
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(epochs):
        model.train()
        for X, y in train_dl:
            opt.zero_grad()
            loss_fn(model(X), y).backward()
            opt.step()
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for X, y in test_dl:
            correct += int((model(X).argmax(1) == y.argmax(1)).sum())
            total += len(y)
    return correct / total


def seed_spread(make_model, train_dl, test_dl, seeds: int = SEEDS, **kwargs) -> np.ndarray:
    out = []
    for seed in range(seeds):
        torch.manual_seed(seed)
        out.append(run(make_model(), train_dl, test_dl, **kwargs))
    return np.array(out)


def report(name: str, scores: np.ndarray, versus: np.ndarray | None = None) -> None:
    line = f"{name:<32} {scores.mean():.4f} +/- {scores.std():.4f}   (n={len(scores)})"
    if versus is not None:
        diff = scores.mean() - versus.mean()
        pooled = np.sqrt(scores.var(ddof=1) / len(scores) + versus.var(ddof=1) / len(versus))
        line += f"   {100 * diff:+.2f} pt, {diff / pooled:+.1f} sigma"
    print(line)


DIM = {enc: load(enc, "train").metadata["embed"]["hidden_size"] for enc in ENCODERS}
N_CLASSES = load("dinov2-small", "train").features["label"].num_classes
print("widths:", DIM, "| classes:", N_CLASSES)

### First, look at the data you are about to slice

That `.shuffle(seed=0)` above is not defensive habit, it is load-bearing, and the reason is
worth two minutes because it is the kind of bug that produces a *plausible* wrong answer
rather than a crash.

In [ ]:
dtd = load("dinov2-small", "train")
labels = dtd["label"][:]
shuffled_labels = dtd.dataset.shuffle(seed=0)["label"][:]

print("first 20 labels in the DTD cache:", labels[:20].tolist())
print()
print(f"{'rows taken':>12} {'classes, as stored':>20} {'classes, shuffled':>19}")
for n in (470, 940, 1880):
    print(f"{n:>12} {len(set(labels[:n].tolist())):>14}/{N_CLASSES:<5} "
          f"{len(set(shuffled_labels[:n].tolist())):>13}/{N_CLASSES}")

**The DTD cache is stored grouped by label.** Take the first 940 rows and you do not get a
smaller sample of the task — you get a *different, easier* task with a quarter of the
classes, and a test set full of categories your model has never once seen.

The reason this is dangerous rather than merely wrong is that it fails *quietly and
plausibly*. Subsampling this way at 470 / 940 / 1880 rows produces test accuracies of
0.12 / 0.24 / 0.44, which looks exactly like a healthy learning curve — right up until you
notice those numbers are just `n / 3760`. The model was scoring near-perfectly on the
classes it had and zero on the rest. Shuffled, the same sizes give 0.69 / 0.75 / 0.78, which
is the real curve and a far less exciting one.

Two general lessons, both cheap:

- **Not every cache is shuffled.** The EuroSAT caches are (they were subsampled at build
  time, which shuffles); the DTD ones were built `--full`, which does not. Do not assume —
  it is one line to check, as above.
- **A number that looks like a textbook result deserves more suspicion, not less.** The
  broken version of this exercise produced a *nicer* curve than the correct one.

## E1 — The baseline, first

**Before you search anything**, establish the two-line baseline on DTD and its seed
spread. A search without a baseline is unreadable; a baseline without a spread cannot tell
you whether a later "win" is real.

This one is done for you, because it is the habit the rest of the notebook depends on.
Every number you produce below gets compared against these two.

In [ ]:
train_dl, test_dl = loaders("dinov2-small")

baseline = seed_spread(lambda: Serial([NeuralNet(hidden_size=DIM["dinov2-small"], out_size=N_CLASSES)]),
                       train_dl, test_dl)
report("baseline: NeuralNet(384, 47)", baseline)
print(f"\n=> a later result must clear {baseline.mean() + 2 * baseline.std():.4f} "
      f"before you should believe it beat this.")

**Write down both numbers now**, in your report, before you have seen anything else. This
is not ceremony — it is the only way to stop yourself from quietly redefining what counted
as success once you have seen the results.

## E2 — Argue from the metadata, then check

Notebook 01 said the metadata is the public API of a cache. Use it as evidence.

**Before running the next cell**, answer in your report:

1. Which of the three encoders should do best on *textures*, and why? Reason from what
   each model was trained to do, not from what you remember of the table in notebook 01.
2. `mobilenet_v2` produces 1280 numbers per image against DINOv2's 384. Does the wider
   representation help here? State the sign *and* a rough size before you look.
3. How big do you expect the gap between best and worst to be — one point, five, fifteen?

Then run it.

In [ ]:
results = {}
for enc in ENCODERS:
    tr, te = loaders(enc)
    torch.manual_seed(0)
    results[enc] = run(Serial([NeuralNet(hidden_size=DIM[enc], out_size=N_CLASSES)]), tr, te)

print(f"{'encoder':<24} {'width':>6} {'accuracy':>9}")
print("-" * 42)
for enc, acc in sorted(results.items(), key=lambda kv: -kv[1]):
    print(f"{enc:<24} {DIM[enc]:>6} {acc:>9.4f}")

spread = max(results.values()) - min(results.values())
print(f"\nspread across encoders: {100 * spread:.1f} pt")
print(f"for comparison, the baseline's own seed spread: {100 * baseline.std():.2f} pt")

Compare that last pair of numbers. **How many seed-spreads wide is the encoder axis?** That
ratio is the honest way to say "this axis matters and that one does not", and it is the
sentence your report should contain.

## E3 — Design and run your own search

Now your search space. Notebook 04's was handed to you; this one is yours to defend.

Edit `param_space` below. Some things worth deciding, and *writing down the reasoning for*:

- **Should `encoder` be in the space at all?** You have just measured that axis directly.
  Leaving it in means most of your budget gets spent re-confirming what you already know.
  Taking it out means your search can only explore the flat part. Which is the better use
  of 16 trials? There is a defensible answer either way — pick one and say why.
- **What is the widest hidden layer worth trying?** Notebook 04's evidence says added depth
  loses on a frozen encoder. Is your space designed to *find* a good architecture, or to
  *test* that claim? Those want different ranges.
- **How wide a learning-rate range?** Too narrow and you have assumed the answer; too wide
  and you burn trials on configs that cannot train.

In [ ]:
from ray import tune
from ray.tune.search.hyperopt import HyperOptSearch


def trainable(config):
    encoder = config["encoder"]
    train = VectorCache.from_hub(f"pttrn-io/dtd-{encoder}", split="train")
    test = VectorCache.from_hub(f"pttrn-io/dtd-{encoder}", split="test")
    n_classes = train.features["label"].num_classes
    dim = train.metadata["embed"]["hidden_size"]

    oh = OneHot(num_classes=n_classes, label_col="label", target_col="onehot")
    col = Collate(embedding_col="embed", target_col="onehot", padder=torch.stack)
    tr = DataLoader(train.dataset.map(oh), batch_size=64, shuffle=True, collate_fn=col)
    te = DataLoader(test.dataset.map(oh), batch_size=64, shuffle=False, collate_fn=col)

    layers = [GaussianNoise(sigma=config["sigma"])]
    if config["width"]:
        layers += [NeuralNet(hidden_size=dim, out_size=config["width"]),
                   NeuralNet(hidden_size=config["width"], out_size=n_classes)]
    else:
        layers += [NeuralNet(hidden_size=dim, out_size=n_classes)]

    torch.manual_seed(0)
    model = Serial(layers)
    opt = torch.optim.Adam(model.parameters(), lr=config["lr"])
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(config["epochs"]):
        model.train()
        for X, y in tr:
            opt.zero_grad()
            loss_fn(model(X), y).backward()
            opt.step()

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for X, y in te:
            correct += int((model(X).argmax(1) == y.argmax(1)).sum())
            total += len(y)
    tune.report({"Accuracy": correct / total})


# ---- YOUR SEARCH SPACE. Edit this, and justify every line of it in your report. ----
param_space = {
    "encoder": tune.choice(["dinov2-small"]),
    "width": tune.choice([0, 64, 256]),
    "sigma": tune.uniform(0.0, 0.5),
    "lr": tune.loguniform(3e-4, 3e-3),
    "epochs": EPOCHS,
}

In [ ]:
tuner = tune.Tuner(
    trainable,
    param_space=param_space,
    tune_config=tune.TuneConfig(metric="Accuracy", mode="max",
                                search_alg=HyperOptSearch(), num_samples=N_TRIALS),
    run_config=tune.RunConfig(storage_path=Path("ray_results").resolve(),
                              name="my_search", verbose=0),
)
start = time.time()
search = tuner.fit()
df = search.get_dataframe()
df = df[["Accuracy"] + [c for c in df.columns if c.startswith("config/") and "epochs" not in c]]
df.columns = [c.replace("config/", "") for c in df.columns]

print(f"{N_TRIALS} trials in {time.time() - start:.0f}s\n")
print(df.sort_values("Accuracy", ascending=False).head(6).to_string(index=False))
print(f"\nbest trial : {df['Accuracy'].max():.4f}")
print(f"baseline   : {baseline.mean():.4f} +/- {baseline.std():.4f}")

**Did anything beat the baseline? By how much?**

Careful with the comparison you just printed. The best trial is a **single seed**, selected
as the maximum of 16 draws. The baseline is a **mean over several seeds**. Taking the max
of many noisy numbers and comparing it to an average is one of the most common ways to
manufacture a win that is not there — the max of 16 draws from the baseline's own
distribution would already sit above the baseline's mean.

Which is exactly what E4 is for.

## E4 — Rerun the winner honestly

Take the winning configuration and rerun it across seeds. Then compare like with like.

In [ ]:
best = search.get_best_result().config
print("winning config:", {k: v for k, v in best.items() if k != "epochs"})


def make_winner():
    dim = DIM[best["encoder"]]
    layers = [GaussianNoise(sigma=best["sigma"])]
    if best["width"]:
        layers += [NeuralNet(hidden_size=dim, out_size=best["width"]),
                   NeuralNet(hidden_size=best["width"], out_size=N_CLASSES)]
    else:
        layers += [NeuralNet(hidden_size=dim, out_size=N_CLASSES)]
    return Serial(layers)


win_tr, win_te = loaders(best["encoder"])
winner = seed_spread(make_winner, win_tr, win_te, lr=best["lr"])

print()
report("baseline", baseline)
report("search winner", winner, versus=baseline)
print(f"\nsingle-seed best from the search : {df['Accuracy'].max():.4f}")
print(f"the same config over {SEEDS} seeds     : {winner.mean():.4f}")

**This is the number that goes in your report**, not the leaderboard number.

Answer these in writing:

1. How much of the apparent win survived being rerun? If it shrank, say by how much.
2. Is the difference larger than the pooled sigma printed above? If not, what is the honest
   one-sentence conclusion?
3. Most "wins" at this scale are inside seed noise. Is yours? Commit to an answer.

## E5 — An ablation you can actually settle

`GaussianNoise` is the one head-side knob with any prior evidence behind it: earlier
measurements had it worth about +0.4 points on DTD. That is small enough that it is a
genuinely open question rather than a known result.

**Predict first:** does input noise still help once the training set is full size? Argue
from what noise is *for* — then test it at two training sizes.

In [ ]:
FULL = 3760


def with_noise(sigma):
    return lambda: Serial([GaussianNoise(sigma=sigma),
                           NeuralNet(hidden_size=DIM["dinov2-small"], out_size=N_CLASSES)])


def train_accuracy(model, train_dl) -> float:
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for X, y in train_dl:
            correct += int((model(X).argmax(1) == y.argmax(1)).sum())
            total += len(y)
    return correct / total


for n_train in (param(940, test=200), None):
    tr, te = loaders("dinov2-small", n_train=n_train)
    rows = n_train or FULL
    # scale epochs so every training size gets a comparable number of gradient steps,
    # otherwise you are measuring "less data" and "less training" at the same time
    epochs = max(EPOCHS, round(EPOCHS * FULL / rows))

    print(f"\n--- {rows} rows, {epochs} epochs ---")
    clean = seed_spread(with_noise(0.0), tr, te, epochs=epochs)
    noisy = seed_spread(with_noise(0.3), tr, te, epochs=epochs)
    report("sigma = 0.0", clean)
    report("sigma = 0.3", noisy, versus=clean)

    torch.manual_seed(0)
    fitted = with_noise(0.0)()
    run(fitted, tr, te, epochs=epochs)
    print(f"{'training-set accuracy':<32} {train_accuracy(fitted, tr):.4f}")

Note the epoch scaling in that cell. Without it you would be changing two things at once —
less data *and* fewer gradient steps — and could not attribute the difference to either.
Say in your report whether you think that was the right call.

Three things to take from the output, and the third is the one worth keeping.

**Check whether either difference cleared its sigma.** If it did not, the honest sentence is
"no measurable effect at this training size", not "noise helped slightly". Resist the pull
of the sign: a +0.1 pt difference at 0.4 sigma is not a small benefit, it is no evidence.

**"Does GaussianNoise help?" is not a question with an answer.** "Does it help at *this*
training size, on *this* representation?" is. Regularisation is supposed to buy something
exactly when a model can memorise its training set — so look at the last line.

**The head reaches ~100% training accuracy and the test number does not move.** That is the
informative result. The model is memorising completely, and regularising it harder changes
nothing downstream, which means the ceiling you are hitting is not head capacity — it is
what the frozen representation makes linearly separable. You cannot regularise your way to
a better representation. That is the same conclusion notebook 04's search reached from the
other direction, and it is worth writing in your report in those terms.

## E6 — Your own hypothesis

One more, of your own choosing. Something you can state as a claim that could fail, build
in a few lines, and measure against the baseline you established in E1. Some that are
cheap and genuinely open:

- Does the encoder ranking from E2 hold on a *different* dataset (`pttrn-io/eurosat-*`), or
  is it a property of textures specifically?
- Does fusion (notebook 04 §4.3) behave differently when the two encoders are close in
  strength — say `resnet-18` + `mobilenet_v2` — than when one clearly dominates?
- Does the gap between the encoders widen or narrow as you shrink the training set? Which
  direction did you expect, and why?
- Does `NeuralNet`'s hidden layer ever pay for itself if you give it far more epochs?

Pick one. State it so it could fail. Measure it over seeds.

---

## The deliverable

Half a page, containing:

1. **The baseline**, with its seed spread, quoted from E1 — and stated to have been
   established before anything else.
2. **Your search space, and why.** Which axes you included, which you deliberately left
   out, and what that cost you.
3. **What the search found**, compared honestly: the winner rerun over seeds, not the
   leaderboard maximum.
4. **At least one negative or null result you produced**, with its numbers. "I expected X,
   measured Y, and here is the spread that convinces me the difference is not real" is a
   complete and respectable finding.
5. **One sentence on what you would measure next**, and why that is the question with the
   most leverage on it.

What is being graded is point 4 and the honesty of point 3. A report where everything
worked is a report where nothing was risked — and, more often than not, a report where the
seed spread was never checked.